# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 6

In this version, we extend the neural character language model with a simple recurrent neural network.

The model keeps the same character vocabulary, fixed context window and trainable embeddings introduced in Version 5.

Instead of concatenating all context embeddings into a single vector, the embeddings are now processed one character at a time.

A recurrent hidden state carries information from one character to the next and provides a learned representation of the sequence.

This version introduces recurrent sequence processing while keeping the model small and easy to inspect.

## 1. Imports and Configuration

The Python standard library configures the execution environment before TensorFlow is imported.

GPU execution is disabled because this small model runs efficiently on the CPU and does not require CUDA. Low-level TensorFlow logs are suppressed to keep the notebook output clean.

TensorFlow provides tensor operations, trainable variables and automatic differentiation.

NumPy remains useful for reproducible data shuffling and sampling, while TensorFlow performs the model calculations and training.

The configuration collects the values that control the experiment.

`CONTEXT_LENGTH` defines how many previous characters are used to predict the next character.

`EMBEDDING_DIM` defines the size of the trainable vector used to represent each character.

`HIDDEN_DIM` defines the size of the recurrent hidden state that carries information through the context.

An explicit seed makes weight initialization, data splitting, mini-batch shuffling and text generation reproducible.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf

tf.config.set_visible_devices([], "GPU")

In [2]:
SEED = 42
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 100

CONTEXT_LENGTH = 4
EMBEDDING_DIM = 8
HIDDEN_DIM = 16

tf.keras.utils.set_random_seed(SEED)

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character patterns.

The corpus is kept unchanged from Version 5 so that the effect of recurrent sequence processing can be observed without changing the training data.

In [3]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.
The neural model also assigns an integer identifier to every character so that characters can be represented numerically.

As in Version 5, these identifiers are used to look up trainable embedding vectors.

In [4]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Context windows and numerical encoding

Each character is converted into its integer identifier.

For the text `modeling`:

`modeling`  
↓  
`[15, 17, 7, 8, 14, 12, 16, 10]`

The model keeps the fixed four-character context introduced in Version 5.

With a context length of 4, four adjacent identifiers form each input context:

- `mode -> l` becomes `[15, 17, 7, 8] -> 14`
- `odel -> i` becomes `[17, 7, 8, 14] -> 12`
- `deli -> n` becomes `[7, 8, 14, 12] -> 16`
- `elin -> g` becomes `[8, 14, 12, 16] -> 10`

The four identifiers in each context are the input. The following identifier is the target to predict.

Unlike Version 5, the four context embeddings will not be concatenated into a single flattened representation.

Instead, they will be processed in order by a recurrent hidden state that carries information from one character to the next.

During generation, predicted identifiers are converted back into characters:

`[15, 17, 7, 8, 14, 12, 16, 10]`  
↓  
`modeling`

In [5]:
examples = [
    (corpus[index:index + CONTEXT_LENGTH], corpus[index + CONTEXT_LENGTH])
    for index in range(len(corpus) - CONTEXT_LENGTH)
]

input_ids = np.array([
    [character_to_id[character] for character in context]
    for context, _ in examples
], dtype=np.int32)

target_ids = np.array([
    character_to_id[target]
    for _, target in examples
], dtype=np.int32)

print("Number of examples:", len(examples))
print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)
print("First 8 examples:", examples[:8])
print("First 8 input IDs:")
print(input_ids[:8])
print("First 8 target IDs:", target_ids[:8])

Number of examples: 436
Input shape: (436, 4)
Target shape: (436,)
First 8 examples: [('lang', 'u'), ('angu', 'a'), ('ngua', 'g'), ('guag', 'e'), ('uage', ' '), ('age ', 'm'), ('ge m', 'o'), ('e mo', 'd')]
First 8 input IDs:
[[14  4 16 10]
 [ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]]
First 8 target IDs: [22  4 10  8  1 15 17  7]


## 3. Neural Model

### Trainable embeddings and recurrent weights

Version 6 replaces the flattened context representation from Version 5 with a simple recurrent neural network.

Each character identifier still selects a trainable embedding vector.

For a context of four characters:

`[15, 17, 7, 8]`  
↓  
`4 embedding vectors`

The embeddings are now processed one at a time and in order.

At every position, the model combines:

- the current character embedding;
- the hidden state produced by the previous position.

The result passes through `tanh` to create a new hidden state:

`current embedding + previous hidden state`  
↓  
`new hidden state`

The hidden state therefore carries information forward through the context.

After the final character, the last hidden state is multiplied by an output weight matrix to produce one logit for every possible next character.

The model learns four parameter matrices:

- the character embedding matrix;
- the input-to-hidden weight matrix;
- the hidden-to-hidden recurrent weight matrix;
- the hidden-to-output weight matrix.

No bias terms are introduced yet, keeping the recurrent mechanism small and easy to inspect.

In [6]:
inputs = tf.convert_to_tensor(input_ids, dtype=tf.int32)

model_random = tf.random.Generator.from_seed(SEED)

embedding_matrix = tf.Variable(
    model_random.normal(
        shape=(vocabulary_size, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

input_weights = tf.Variable(
    model_random.normal(
        shape=(EMBEDDING_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

recurrent_weights = tf.Variable(
    model_random.normal(
        shape=(HIDDEN_DIM, HIDDEN_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

output_weights = tf.Variable(
    model_random.normal(
        shape=(HIDDEN_DIM, vocabulary_size),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)


def recurrent_forward(
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    inputs
):
    embeddings = tf.gather(embedding_matrix, inputs)

    hidden_state = tf.zeros(
        (tf.shape(inputs)[0], HIDDEN_DIM),
        dtype=tf.float32
    )

    for position in range(CONTEXT_LENGTH):
        current_embedding = embeddings[:, position, :]

        hidden_state = tf.tanh(
            tf.matmul(current_embedding, input_weights)
            +
            tf.matmul(hidden_state, recurrent_weights)
        )

    logits = tf.matmul(hidden_state, output_weights)

    return hidden_state, logits


example_embeddings = tf.gather(embedding_matrix, inputs[:1])

example_hidden_state, example_logits = recurrent_forward(
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    inputs[:1]
)

trainable_parameters = (
    tf.size(embedding_matrix)
    + tf.size(input_weights)
    + tf.size(recurrent_weights)
    + tf.size(output_weights)
)

print("Input tensor shape:", inputs.shape)
print("Embedding matrix shape:", embedding_matrix.shape)
print("Embedded context shape:", example_embeddings.shape)
print("Input weight matrix shape:", input_weights.shape)
print("Recurrent weight matrix shape:", recurrent_weights.shape)
print("Final hidden state shape:", example_hidden_state.shape)
print("Output weight matrix shape:", output_weights.shape)
print("Logits shape:", example_logits.shape)
print("Trainable parameters:", trainable_parameters.numpy())

Input tensor shape: (436, 4)
Embedding matrix shape: (27, 8)
Embedded context shape: (1, 4, 8)
Input weight matrix shape: (8, 16)
Recurrent weight matrix shape: (16, 16)
Final hidden state shape: (1, 16)
Output weight matrix shape: (16, 27)
Logits shape: (1, 27)
Trainable parameters: 1032


### Training and validation split

The examples are divided into two separate groups:

- the training set is used to update the model parameters;
- the validation set is used to measure the loss on examples that do not update the parameters.

Each input example contains an ordered sequence of four character identifiers.

The recurrent model processes these identifiers from left to right. The hidden state starts from zeros for every context example and is updated once for each character.

The indices are shuffled with a local random generator, making the split reproducible.

In [7]:
split_random = np.random.default_rng(SEED)

indices = split_random.permutation(len(inputs))
split_position = int(len(indices) * TRAIN_FRACTION)

train_indices = indices[:split_position]
validation_indices = indices[split_position:]

train_inputs = tf.gather(inputs, train_indices)
train_targets = tf.gather(target_ids, train_indices)

validation_inputs = tf.gather(inputs, validation_indices)
validation_targets = tf.gather(target_ids, validation_indices)

print("Training examples:", len(train_inputs))
print("Validation examples:", len(validation_inputs))
print("Training input shape:", train_inputs.shape)
print("Validation input shape:", validation_inputs.shape)

Training examples: 348
Validation examples: 88
Training input shape: (348, 4)
Validation input shape: (88, 4)


### Softmax probabilities

TensorFlow provides `tf.nn.softmax` to convert logits into probabilities.

- every probability is between 0 and 1;
- the probabilities for one input sum to 1;
- higher logits produce higher probabilities.

TensorFlow handles the numerical stability of this operation internally.

In [8]:
def softmax(logits):
    return tf.nn.softmax(logits, axis=1)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

Before calculating the loss, the context identifiers are mapped to their embedding vectors.

The embeddings are processed sequentially by the recurrent network. At every position, the current embedding and the previous hidden state are combined to produce a new hidden state.

After the final context character, the last hidden state is multiplied by the output weights to produce the logits.

TensorFlow calculates the cross-entropy directly from the logits and integer target IDs using a numerically stable operation.

A high probability for the correct character produces a low loss.

Training will update the embedding matrix, input weights, recurrent weights and output weights to reduce this value.

In [9]:
def calculate_loss(
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    inputs,
    targets
):
    _, logits = recurrent_forward(
        embedding_matrix,
        input_weights,
        recurrent_weights,
        output_weights,
        inputs
    )

    example_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(example_losses)

In [10]:
initial_train_loss = calculate_loss(
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    train_inputs,
    train_targets
)

initial_validation_loss = calculate_loss(
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    validation_inputs,
    validation_targets
)

print("Initial train loss:", initial_train_loss.numpy())
print("Initial validation loss:", initial_validation_loss.numpy())

Initial train loss: 3.2958367
Initial validation loss: 3.2958395


### Training with mini-batch gradient descent

Training remains organized into epochs.

During every epoch:

1. the training examples are shuffled;
2. the examples are divided into mini-batches;
3. `tf.GradientTape` records the recurrent forward calculations;
4. TensorFlow calculates the gradients automatically;
5. the embedding matrix, input weights, recurrent weights and output weights are updated with gradient descent;
6. training and validation loss are measured.

The validation examples are never used to update the model parameters.

The recurrent computations are recorded by `tf.GradientTape`. Gradients therefore flow backward through the sequence of hidden-state updates.

The same recurrent weight matrix is reused at every context position, allowing the model to learn how information should be carried from one character to the next.

Version 6 keeps the same mini-batch training procedure introduced in the previous versions while extending automatic differentiation to all four trainable parameter matrices.

### Mini-batches

A mini-batch is a small group of training examples.

The model updates its parameters after every mini-batch instead of processing all training examples together.

The final mini-batch may contain fewer examples than the configured batch size.

In [11]:
def create_batches(inputs, targets, batch_size):
    for start in range(0, len(inputs), batch_size):
        end = start + batch_size

        batch_inputs = inputs[start:end]
        batch_targets = targets[start:end]

        yield batch_inputs, batch_targets

In [12]:
def train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_embedding_matrix,
    initial_input_weights,
    initial_recurrent_weights,
    initial_output_weights,
    learning_rate=1.0,
    batch_size=32,
    epochs=100,
    seed=42,
    print_every=10
):
    trained_embedding_matrix = tf.Variable(initial_embedding_matrix)
    trained_input_weights = tf.Variable(initial_input_weights)
    trained_recurrent_weights = tf.Variable(initial_recurrent_weights)
    trained_output_weights = tf.Variable(initial_output_weights)

    training_random = np.random.default_rng(seed)

    train_loss_history = []
    validation_loss_history = []

    for epoch in range(epochs + 1):
        train_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_input_weights,
                trained_recurrent_weights,
                trained_output_weights,
                train_inputs,
                train_targets
            ).numpy()
        )

        validation_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_input_weights,
                trained_recurrent_weights,
                trained_output_weights,
                validation_inputs,
                validation_targets
            ).numpy()
        )

        train_loss_history.append(train_loss)
        validation_loss_history.append(validation_loss)

        if print_every is not None and epoch % print_every == 0:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Validation loss: {validation_loss:.4f}"
            )

        if epoch == epochs:
            break

        shuffled_indices = training_random.permutation(len(train_inputs))

        shuffled_inputs = tf.gather(train_inputs, shuffled_indices)
        shuffled_targets = tf.gather(train_targets, shuffled_indices)

        for batch_inputs, batch_targets in create_batches(
            shuffled_inputs,
            shuffled_targets,
            batch_size
        ):
            with tf.GradientTape() as tape:
                batch_loss = calculate_loss(
                    trained_embedding_matrix,
                    trained_input_weights,
                    trained_recurrent_weights,
                    trained_output_weights,
                    batch_inputs,
                    batch_targets
                )

            (
                embedding_gradient,
                input_gradient,
                recurrent_gradient,
                output_gradient
            ) = tape.gradient(
                batch_loss,
                [
                    trained_embedding_matrix,
                    trained_input_weights,
                    trained_recurrent_weights,
                    trained_output_weights
                ]
            )

            trained_embedding_matrix.assign_sub(learning_rate * tf.convert_to_tensor(embedding_gradient))

            trained_input_weights.assign_sub(learning_rate * input_gradient)

            trained_recurrent_weights.assign_sub(learning_rate * recurrent_gradient)

            trained_output_weights.assign_sub(learning_rate * output_gradient)

    return (
        trained_embedding_matrix,
        trained_input_weights,
        trained_recurrent_weights,
        trained_output_weights,
        train_loss_history,
        validation_loss_history
    )

In [13]:
(
    trained_embedding_matrix,
    trained_input_weights,
    trained_recurrent_weights,
    trained_output_weights,
    train_loss_history,
    validation_loss_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED
)

final_train_loss = train_loss_history[-1]
final_validation_loss = validation_loss_history[-1]

best_validation_epoch = int(np.argmin(validation_loss_history))
best_validation_loss = validation_loss_history[best_validation_epoch]

print()
print("Initial train loss:", initial_train_loss.numpy())
print("Final train loss:", final_train_loss)
print("Initial validation loss:", initial_validation_loss.numpy())
print("Final validation loss:", final_validation_loss)
print("Best validation loss:", best_validation_loss)
print("Best validation epoch:", best_validation_epoch)

Epoch   0 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  10 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  20 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  30 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  40 | Train loss: 3.2957 | Validation loss: 3.2958
Epoch  50 | Train loss: 3.2833 | Validation loss: 3.2915
Epoch  60 | Train loss: 2.6701 | Validation loss: 2.9969
Epoch  70 | Train loss: 2.4947 | Validation loss: 3.1296
Epoch  80 | Train loss: 2.1892 | Validation loss: 3.1239
Epoch  90 | Train loss: 1.8286 | Validation loss: 2.9522
Epoch 100 | Train loss: 1.5251 | Validation loss: 3.0352

Initial train loss: 3.2958367
Final train loss: 1.525100827217102
Initial validation loss: 3.2958395
Final validation loss: 3.0351641178131104
Best validation loss: 2.8396050930023193
Best validation epoch: 92


### Reading the losses

The initial training and validation losses are both approximately `3.2958`, which is close to `ln(27)` and therefore consistent with an almost uniform prediction over the vocabulary.

During the first part of training, the recurrent model learns slowly. The embedding vectors and recurrent weight matrices are initialized with small values, so the initial hidden states and parameter updates are also small.

After approximately 50 epochs, the training loss begins to decrease more clearly.

The training loss decreases from approximately:

`3.2958 -> 1.5251`

The validation loss also improves during training and reaches its minimum at epoch 92:

`Best validation loss: 2.8396`

After this point, the validation loss increases slightly while the training loss continues to decrease.

This indicates the beginning of overfitting, although it is much less pronounced than in Version 5 after 100 epochs.

The notebook records the best validation epoch for analysis but intentionally keeps the final epoch-100 parameters for the following demonstrations.

This experiment shows that the recurrent architecture can learn ordered character information while keeping approximately the same model size as Version 5.

### Learned probabilities

After training, the model can produce a probability distribution for the next character from an ordered four-character context.

The character identifiers are mapped to embeddings and processed sequentially by the recurrent network.

The final hidden state summarizes the information carried through the context.

This hidden state is converted into logits and then into probabilities with softmax.

The example below inspects the learned next-character distribution for the context `mode`.

In [14]:
example_context = "mode"

example_context_ids = tf.constant(
    [[character_to_id[character] for character in example_context]],
    dtype=tf.int32
)

example_hidden_state, example_logits = recurrent_forward(
    trained_embedding_matrix,
    trained_input_weights,
    trained_recurrent_weights,
    trained_output_weights,
    example_context_ids
)

learned_probabilities = softmax(example_logits)[0].numpy()

sorted_probabilities = sorted(
    zip(vocabulary, learned_probabilities),
    key=lambda item: item[1],
    reverse=True
)

print("Context:", repr(example_context))
print("Final hidden state shape:", example_hidden_state.shape)
print()

for character, probability in sorted_probabilities:
    print(repr(character), round(float(probability), 4))

print()
print("Total probability:", learned_probabilities.sum())

Context: 'mode'
Final hidden state shape: (1, 16)

' ' 0.5088
'l' 0.1632
'i' 0.0957
'a' 0.0596
's' 0.0581
't' 0.0311
'm' 0.0268
'd' 0.0129
'g' 0.0092
'u' 0.0092
',' 0.0043
'x' 0.0041
'p' 0.0038
'c' 0.0034
'r' 0.0032
'e' 0.002
'.' 0.0014
'y' 0.0011
'n' 0.001
'v' 0.0006
'k' 0.0003
'w' 0.0001
'b' 0.0
'h' 0.0
'o' 0.0
'f' 0.0
'\n' 0.0

Total probability: 1.0


## 4. Generator

The trained recurrent model can now generate new text one character at a time.

Generation remains autoregressive.

For every prediction:

1. the four most recent characters form the current context;
2. their identifiers are mapped to trainable embeddings;
3. the embeddings are processed from left to right by the recurrent network;
4. the final hidden state produces the next-character logits;
5. softmax converts the logits into probabilities;
6. one character is sampled from the probability distribution;
7. the sampled character is appended to the generated text;
8. the four-character context window moves forward.

The recurrent hidden state is initialized from zeros for each context window, matching the way the model was trained.

A local NumPy random generator keeps text generation reproducible without changing the global random state.

### Sample the next character

The current four-character context is converted into numerical identifiers and processed by the recurrent model.

The final hidden state is transformed into logits and probabilities.

A local random generator samples one identifier from this learned probability distribution and converts it back into a character.

In [15]:
def sample_next_character(
    context,
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    random_generator
):
    context_ids = tf.constant(
        [[character_to_id[character] for character in context]],
        dtype=tf.int32
    )

    _, logits = recurrent_forward(
        embedding_matrix,
        input_weights,
        recurrent_weights,
        output_weights,
        context_ids
    )

    probabilities = softmax(logits)[0].numpy()

    next_id = random_generator.choice(vocabulary_size, p=probabilities)

    return id_to_character[next_id]

In [16]:
sample_random = np.random.default_rng(SEED)

example_context = "mode"

print("Context:", repr(example_context))

for _ in range(5):
    sampled_character = sample_next_character(
        example_context,
        trained_embedding_matrix,
        trained_input_weights,
        trained_recurrent_weights,
        trained_output_weights,
        sample_random
    )

    print("Sampled character:", repr(sampled_character))

Context: 'mode'
Sampled character: 'l'
Sampled character: ' '
Sampled character: 'l'
Sampled character: 'k'
Sampled character: ' '


### Generate text

Text generation starts from a four-character context.

At every step, the model processes the current context recurrently and samples the next character.

The sampled character is appended to the output.

The oldest context character is then removed, causing the fixed context window to slide forward by one position.

For example:

`mode -> sampled character`

then:

`ode? -> next sampled character`

and so on.

The model therefore generates text autoregressively while using recurrent processing inside every context window.

In [17]:
def generate_text(
    starting_context,
    number_of_characters,
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    seed=42
):
    if len(starting_context) != CONTEXT_LENGTH:
        raise ValueError(f"starting_context must contain exactly {CONTEXT_LENGTH} characters")

    generated_text = starting_context

    generation_random = np.random.default_rng(seed)

    for _ in range(number_of_characters):
        current_context = generated_text[-CONTEXT_LENGTH:]

        next_character = sample_next_character(
            current_context,
            embedding_matrix,
            input_weights,
            recurrent_weights,
            output_weights,
            generation_random
        )

        generated_text += next_character

    return generated_text

In [18]:
generated_text = generate_text(
    starting_context="mode",
    number_of_characters=300,
    embedding_matrix=trained_embedding_matrix,
    input_weights=trained_input_weights,
    recurrent_weights=trained_recurrent_weights,
    output_weights=trained_output_weights,
    seed=SEED
)

print(generated_text)

model,er wote more le the uoral eriteamade nestars er ibspirn cerdedps.
.
,ele meepte s.
 pg.
cle mead.nl iseane teanimine ter male inctean wem meune leactucs,edllmas.
.
.
,u,n ve nuurspheerd texh.
.
.
,eeunc.
.
 fhe the inginpses mocdexturniccvomeinine toralion math wace be coune to made bed texdosuorn


## 5. Tests

These assertions verify the context windows, TensorFlow tensors, trainable recurrent parameters, data split, model shapes, hidden states, probability distributions, gradients, training behavior and reproducibility.

The tests verify that:

- context windows and targets are constructed correctly;
- all four parameter matrices are trainable TensorFlow variables;
- embeddings, recurrent states, logits and parameter matrices have the expected shapes;
- hidden-state values remain finite and inside the range produced by `tanh`;
- training and validation sets remain separate;
- probability distributions sum to 1;
- TensorFlow produces finite gradients for every trainable parameter matrix;
- all trainable parameters actually change during training;
- repeating the complete training procedure with the same seed produces the same parameters and loss histories;
- autoregressive generation is reproducible with the same context and seed.

The tests also verify that generation is reproducible when the same starting context and seed are used.

In [19]:
assert len(examples) == len(corpus) - CONTEXT_LENGTH
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)

assert input_ids.shape == (len(examples), CONTEXT_LENGTH)
assert target_ids.shape == (len(examples),)

assert examples[0][0] == corpus[:CONTEXT_LENGTH]
assert examples[0][1] == corpus[CONTEXT_LENGTH]

assert examples[1][0] == corpus[1:1 + CONTEXT_LENGTH]
assert examples[1][1] == corpus[1 + CONTEXT_LENGTH]

assert tf.is_tensor(inputs)

assert isinstance(embedding_matrix, tf.Variable)
assert isinstance(input_weights, tf.Variable)
assert isinstance(recurrent_weights, tf.Variable)
assert isinstance(output_weights, tf.Variable)

assert isinstance(trained_embedding_matrix, tf.Variable)
assert isinstance(trained_input_weights, tf.Variable)
assert isinstance(trained_recurrent_weights, tf.Variable)
assert isinstance(trained_output_weights, tf.Variable)

assert inputs.shape == (len(examples), CONTEXT_LENGTH)

assert embedding_matrix.shape == (vocabulary_size, EMBEDDING_DIM)

assert input_weights.shape == (EMBEDDING_DIM, HIDDEN_DIM)

assert recurrent_weights.shape == (HIDDEN_DIM, HIDDEN_DIM)

assert output_weights.shape == (HIDDEN_DIM, vocabulary_size)

assert trained_embedding_matrix.shape == embedding_matrix.shape
assert trained_input_weights.shape == input_weights.shape
assert trained_recurrent_weights.shape == recurrent_weights.shape
assert trained_output_weights.shape == output_weights.shape

assert trainable_parameters.numpy() == 1032

assert example_embeddings.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)

assert example_hidden_state.shape == (1, HIDDEN_DIM)

assert example_logits.shape == (1, vocabulary_size)

assert np.all(np.isfinite(example_hidden_state.numpy()))
assert np.all(example_hidden_state.numpy() >= -1.0)
assert np.all(example_hidden_state.numpy() <= 1.0)

assert len(train_inputs) + len(validation_inputs) == len(inputs)

assert len(train_inputs) == len(train_targets)
assert len(validation_inputs) == len(validation_targets)

assert len(np.intersect1d(train_indices, validation_indices)) == 0

assert len(train_loss_history) == EPOCHS + 1
assert len(validation_loss_history) == EPOCHS + 1

assert final_train_loss < float(initial_train_loss.numpy())

assert np.all(np.isfinite(train_loss_history))
assert np.all(np.isfinite(validation_loss_history))

probability_context = tf.constant(
    [[character_to_id[character] for character in "mode"]],
    dtype=tf.int32
)

_, probability_logits = recurrent_forward(
    trained_embedding_matrix,
    trained_input_weights,
    trained_recurrent_weights,
    trained_output_weights,
    probability_context
)

probability_values = softmax(probability_logits)[0].numpy()

assert abs(probability_values.sum() - 1.0) < 1e-6
assert np.all(np.isfinite(probability_values))
assert np.all(probability_values >= 0.0)

gradient_inputs = train_inputs[:BATCH_SIZE]
gradient_targets = train_targets[:BATCH_SIZE]

with tf.GradientTape() as tape:
    gradient_loss = calculate_loss(
        embedding_matrix,
        input_weights,
        recurrent_weights,
        output_weights,
        gradient_inputs,
        gradient_targets
    )

(
    embedding_gradient,
    input_gradient,
    recurrent_gradient,
    output_gradient
) = tape.gradient(
    gradient_loss,
    [
        embedding_matrix,
        input_weights,
        recurrent_weights,
        output_weights
    ]
)

assert embedding_gradient is not None
assert input_gradient is not None
assert recurrent_gradient is not None
assert output_gradient is not None

dense_embedding_gradient = tf.convert_to_tensor(embedding_gradient)

assert dense_embedding_gradient.shape == embedding_matrix.shape
assert input_gradient.shape == input_weights.shape
assert recurrent_gradient.shape == recurrent_weights.shape
assert output_gradient.shape == output_weights.shape

assert np.all(np.isfinite(dense_embedding_gradient.numpy()))

assert np.all(np.isfinite(input_gradient.numpy()))

assert np.all(np.isfinite(recurrent_gradient.numpy()))

assert np.all(np.isfinite(output_gradient.numpy()))

(
    repeated_embedding_matrix,
    repeated_input_weights,
    repeated_recurrent_weights,
    repeated_output_weights,
    repeated_train_history,
    repeated_validation_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    input_weights,
    recurrent_weights,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
    print_every=None
)

assert np.allclose(trained_embedding_matrix.numpy(), repeated_embedding_matrix.numpy())

assert np.allclose(trained_input_weights.numpy(), repeated_input_weights.numpy())

assert np.allclose(trained_recurrent_weights.numpy(), repeated_recurrent_weights.numpy())

assert np.allclose(trained_output_weights.numpy(), repeated_output_weights.numpy())

assert np.allclose(train_loss_history, repeated_train_history)

assert np.allclose(validation_loss_history, repeated_validation_history)

assert not np.allclose(embedding_matrix.numpy(), trained_embedding_matrix.numpy())

assert not np.allclose(input_weights.numpy(), trained_input_weights.numpy())

assert not np.allclose(recurrent_weights.numpy(), trained_recurrent_weights.numpy())

assert not np.allclose(output_weights.numpy(), trained_output_weights.numpy())

first_generation = generate_text(
    starting_context="mode",
    number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    input_weights=trained_input_weights,
    recurrent_weights=trained_recurrent_weights,
    output_weights=trained_output_weights,
    seed=10
)

second_generation = generate_text(
    starting_context="mode",
    number_of_characters=30,
    embedding_matrix=trained_embedding_matrix,
    input_weights=trained_input_weights,
    recurrent_weights=trained_recurrent_weights,
    output_weights=trained_output_weights,
    seed=10
)

assert first_generation == second_generation

assert len(first_generation) == (CONTEXT_LENGTH + 30)

assert first_generation.startswith("mode")

print("All checks passed.")

All checks passed.


## Notes

- The model remains a character-level neural language model.
- The training corpus and vocabulary remain unchanged from Version 5.
- Each prediction uses a fixed context of four previous characters.
- Character identifiers are mapped to trainable embedding vectors.
- The context embeddings are processed sequentially instead of being concatenated into a single flattened vector.
- A recurrent hidden state carries information from one context position to the next.
- The hidden state starts from zeros for every training example.
- `tanh` introduces a nonlinear recurrent transformation.
- The model learns four trainable parameter matrices:
  - the embedding matrix;
  - the input-to-hidden weight matrix;
  - the recurrent hidden-to-hidden weight matrix;
  - the hidden-to-output weight matrix.
- The recurrent weights are reused at every position in the context.
- All trainable parameters are optimized automatically with `tf.GradientTape`.
- The model contains 1032 trainable parameters, slightly fewer than the 1080 parameters used in Version 5.
- Training and validation examples remain reproducibly separated.
- Training examples are shuffled at the beginning of every epoch.
- Mini-batches are used to update the model parameters.
- Training loss decreases from approximately `3.2958` to `1.5251`.
- The best validation loss is approximately `2.8396` at epoch 92.
- Validation loss increases slightly after the best epoch, showing the beginning of overfitting.
- The final epoch-100 parameters are intentionally retained for generation.
- Generation remains autoregressive and uses a sliding four-character context window.
- Each generation window is processed recurrently from a zero hidden state, matching the training procedure.
- The same seed reproduces parameter initialization, data splitting, mini-batch shuffling, training and generation.
- The tests verify recurrent states, parameter shapes, probability distributions, gradients, parameter updates and reproducibility.

Version 6 introduces recurrent sequence processing and a hidden state while preserving the fixed four-character contexts introduced in Version 5.

The hidden state carries information through the characters inside each context, but it is not yet preserved across the complete training corpus or indefinitely during generation.

This limitation keeps the recurrent mechanism easy to inspect and prepares future versions for more advanced recurrent architectures such as GRU and LSTM networks.

Future versions will introduce new components and gradually evolve the architecture.